In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
# from huggingface_hub import snapshot_download
# import time

# while True:
#     try:
#         snapshot_download(
#             repo_id="Ken-Z/Latin-Audio", 
#             repo_type="dataset", 
#             local_dir="./Latin-Audio",
#         )
#         break
#     except Exception as e:
#         time.sleep(60 * 5)

In [6]:
!ls Latin-Audio

README.md  metadata.csv  part_0  part_1  part_2


In [8]:
df = pd.read_csv('Latin-Audio/metadata.csv').to_dict(orient = 'records')
df[0]

{'file_name': 'part_0/audio/Seneca, Medea 1-55 (Musa Pedestris)_0000.wav',
 'transcription': 'Dī coniugālīs, tūque geniali stōri.'}

In [11]:
def loop(rows):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    rows, _ = rows

    base = 'Latin-Audio_audio'
    os.makedirs(base, exist_ok=True)
    
    data = []
    for row in tqdm(rows):
        file_name = os.path.join('Latin-Audio', row['file_name'])
        if not os.path.exists(file_name):
            continue

        t = row['transcription'].strip()
        if len(t) < 2:
            continue

        f_new = file_name.replace(' ', '_').replace('/', '-').replace('.wav', '.mp3')
        audio_filename = os.path.join(base, f_new)
        audio_np, sr = sf.read(file_name)
        if audio_np.ndim > 1:
            audio_np = audio_np.mean(axis=1)
        if audio_np.shape[0] < 10000:
            continue
        sf.write(audio_filename, audio_np, sr)
        
        data.append({
            'audio_filename': audio_filename,
            'text': t,
            'speaker': f"{base}"
        })
        
    return data

In [12]:
data = loop((df[:10], 0))

100%|██████████| 10/10 [00:01<00:00,  5.05it/s]


In [14]:
data = multiprocessing(df, loop, cores = 20)

100%|██████████| 1215/1215 [03:06<00:00,  6.53it/s]


In [15]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'Latin-Audio_audio/Latin-Audio-part_0-audio-Seneca,_Medea_1-55_(Musa_Pedestris)_0000.mp3',
 'text': 'Dī coniugālīs, tūque geniali stōri.',
 'speaker': 'Latin-Audio_audio'}

In [16]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'Latin-Audio')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 75.66ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  81%|████████▏ | 1.63MB / 2.01MB,  167kB/s  
Processing Files (1 / 1): 100%|██████████| 2.01MB / 2.01MB,  203kB/s  
Processing Files (1 / 1): 100%|██████████| 2.01MB / 2.01MB,  201kB/s  
New Data Upload: 100%|██████████| 2.01MB / 2.01MB,  201kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:10<00:00, 10.40s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/2a48560be5f28488461708b6684de72d01ad3718', commit_message='Upload dataset', commit_description='', oid='2a48560be5f28488461708b6684de72d01ad3718', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [17]:
audio_files = [d['audio_filename'] for d in data]

a = list(set(audio_files))
with open('Latin-Audio-audio.json', 'w') as fopen:
    json.dump(a, fopen)

pd.DataFrame({'audio': a}).to_parquet('Latin-Audio-audio.parquet')

In [4]:
# !zip -rq Latin-Audio_audio.zip Latin-Audio_audio

In [3]:
# !hf upload malaysia-ai/Multilingual-TTS Latin-Audio_audio.zip --repo-type=dataset

In [7]:
# !zip -rq Latin-Audio_audio_neucodec.zip Latin-Audio_audio_neucodec

In [8]:
# !hf upload malaysia-ai/Multilingual-TTS Latin-Audio_audio_neucodec.zip --repo-type=dataset